# Clean and Process df

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.utils import col_config_og, col_config_sensitivity
import numpy as np
import pandas as pd
import src.clean as clean

pd.set_option("display.max_columns", 500)

In [ ]:
df = pd.read_csv(
    BASE_PATH / "data" / "complete_raw" / "comppsych_dawbas_20210414-0.csv"
)
df.shape

In [ ]:
meta_cols = col_config_og["meta"]

self_impact_cols = col_config_og["self_impact"]
parent_impact_cols = col_config_og["parent_impact"]

self_text_cols = col_config_og["self_text"]
parent_text_cols = col_config_og["parent_text"]

self_quest_cols = col_config_og["self_quest"]
parent_quest_cols = col_config_og["parent_quest"]


all_self_cols = self_quest_cols + self_text_cols + self_impact_cols
all_parent_cols = parent_quest_cols + parent_text_cols + parent_impact_cols
all_depr_cols = all_self_cols + all_parent_cols

## Clean

Remove duplicated SDAN (23717.0) at idx 234 (double the amount of null), same patient

In [ ]:
df_clean = df.drop(index=234)

Remove participants who `removed consent`

In [ ]:
df_clean = df_clean[df_clean["Eligible"] != "Did not or withdrew assent/consent"].copy()

Create new eligibility col to stratify on excluded bc meets criteria, excluded bc not meet criteria, & everything else

In [ ]:
mapping = {
    "Excluded: meets exclusionary criteria (substance use, psychosis, etc.)": "excluded",
    "Excluded: does not meet criteria": "excluded",
    "Excluded: cannot be reached or scheduled, all contact options exhausted": "Include",
    "Include: can't scan (braces, etc.)": "Include",
    ## These were all previously mapped to Include, explicitly coding now for clarity
    "On hold: contact again after specified amount of time": "Include",
    "On hold: low priority": "Include",
    "Include": "Include",
}

df_clean["eligible_group"] = df_clean["Eligible"].map(mapping).fillna("Include")
df_clean["eligible_group"].value_counts(dropna=False)

Fill participant type NAs

In [ ]:
df_clean["Participant_Type2"] = df_clean["Participant_Type2"].fillna("UNSURE")

Subset to only include relevant depression + general cols

In [ ]:
keep_cols = meta_cols + all_depr_cols
df_subset = df_clean[keep_cols]
df_subset.shape

Split

In [ ]:
from sklearn.model_selection import train_test_split
from shutil import rmtree

##### export raw
export_dir = BASE_PATH / "data" / "cleaned" / "initial_split"
if export_dir.exists():
    rmtree(export_dir)
export_dir.mkdir(exist_ok=False, parents=True)
df_subset.to_parquet(export_dir / "raw_clean_depression.parquet")

##### Split
train, test, _, _ = train_test_split(
    df_subset,
    np.ones(len(df_subset)),
    train_size=0.67,
    random_state=SEED,
    shuffle=True,
    stratify=df_subset["eligible_group"],
)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
## Export
train.to_parquet(export_dir / "train.parquet")
test.to_parquet(export_dir / "test.parquet")

## Work only on train set

Cleaning framework:
- Create columns
	- Target impact score as sum of impact questions
	- DAWBA checkpoints (Depressoon, Irritability, Anhedonia) 
- Use DAWBA checkpoint framework to fill un-prompted questionnaire response NAs w/ 0
- Drop healthy participants w/o any of the 3 checkpoints^
- Drop 18i-18l due to large scale missingness
- Fill missing text responses as "Not answered" for participants w/ 5 or less misisng entries
- Drop all remaining participants w/ a missing entry 

In [ ]:
df_sub = pd.read_parquet(
    BASE_PATH / "data" / "cleaned" / "initial_split" / "train.parquet"
)
df_sub.shape

Seperate self & parent dfs

In [ ]:
self_df = df_sub[meta_cols + all_self_cols].copy()
parent_df = df_sub[meta_cols + all_parent_cols].copy()

In [ ]:
# Create Depression/Irritability/Loss of Interest checks
self_df_w_cols = clean.create_checkpoint_cols(
    df=self_df, impact_type="self", impact_cols=self_impact_cols
)
parent_df_w_cols = clean.create_checkpoint_cols(
    df=parent_df, impact_type="parent", impact_cols=parent_impact_cols
)

# Fill missings w/ 0 when not prompted to answer questions, given answers to previous questions
self_df_coded = clean.mark_NA_0_when_not_prompted(
    df=self_df_w_cols, impact_type="self", impact_cols=self_impact_cols
)
parent_df_coded = clean.mark_NA_0_when_not_prompted(
    df=parent_df_w_cols, impact_type="parent", impact_cols=parent_impact_cols
)

# Drop healthy patients to avoid inflated results from easy-to-predict entries
self_df_filter = clean.remove_0_impact_participants(
    df=self_df_coded, impact_type="self", impact_cols=self_impact_cols
)
parent_df_filter = clean.remove_0_impact_participants(
    df=parent_df_coded, impact_type="parent", impact_cols=parent_impact_cols
)

# Clean up remaining NAs + drop heavily missing columns
self_df_final = clean.remove_remaining_missing(
    df=self_df_filter,
    impact_type="self",
    quest_cols=self_quest_cols,
    text_cols=self_text_cols,
)
parent_df_final = clean.remove_remaining_missing(
    df=parent_df_filter,
    impact_type="parent",
    quest_cols=parent_quest_cols,
    text_cols=parent_text_cols,
)

In [ ]:
r_parent = pd.read_parquet(BASE_PATH / "data/cleaned/processed/parent.parquet")
r_self = pd.read_parquet(BASE_PATH / "data/cleaned/processed/self.parquet")

## Export

Export base dfs

In [ ]:
export_dir = BASE_PATH / "data" / "cleaned" / "processed"
if export_dir.exists():
    rmtree(export_dir)
export_dir.mkdir(exist_ok=False, parents=True)
# Self
self_df_sub = self_df_final.reset_index(drop=True)
self_df_sub.to_parquet(export_dir / "self.parquet")
# Parent
parent_df_sub = parent_df_final.reset_index(drop=True)
parent_df_sub.to_parquet(export_dir / "parent.parquet")

# Also process meta info file for easier use later

In [ ]:
from shutil import rmtree

self_text_cols = col_config_sensitivity["self_text"]
parent_text_cols = col_config_sensitivity["parent_text"]
meta_df = pd.read_excel(
    BASE_PATH
    / "data"
    / "complete_raw"
    / "dawba_column_names_and_descriptions_edit.xlsx"
)

In [ ]:
keep_cols = ["new_name", "description"]
self_meta = meta_df[meta_df["new_name"].isin(self_text_cols)][keep_cols]
parent_meta = meta_df[meta_df["new_name"].isin(parent_text_cols)][keep_cols]
## Ensure we got exactly right number of columns
assert len(self_meta) == len(self_text_cols)
assert len(parent_meta) == len(parent_text_cols)

In [ ]:
def strip_prefix(meta_df):
    new_rows_list = []
    for idx, row in meta_df.iterrows():

        og_text = row.description
        og_text_split = og_text.split(")")
        # Get only text after `Dep: text n)` prefix + remove leading/trailing whitespace + remove new lines + remove double spaces
        new_text = (
            " ".join(og_text_split[1:]).strip().replace("\n", " ").replace("  ", " ")
        )
        cur_row = {"question_col": row.new_name, "question_text": new_text}
        new_rows_list.append(cur_row)
    return pd.DataFrame(new_rows_list)

In [ ]:
self_meta_cleaned = strip_prefix(self_meta)
parent_meta_cleaned = strip_prefix(parent_meta)

Export

In [ ]:
meta_export_dir = BASE_PATH / "data" / "cleaned" / "meta_text"
if meta_export_dir.exists():
    rmtree(meta_export_dir)
meta_export_dir.mkdir(exist_ok=False, parents=True)
self_meta_cleaned.to_csv(meta_export_dir / "self.csv")
parent_meta_cleaned.to_csv(meta_export_dir / "parent.csv")